In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

def load_project_env() -> Path:
    """Load the first .env found from the current folder up to the workspace root."""
    for candidate_dir in [Path.cwd(), *Path.cwd().resolve().parents]:
        candidate = candidate_dir / ".env"
        if candidate.exists():
            load_dotenv(candidate, override=True)
            return candidate
    raise FileNotFoundError("No se encontró el archivo .env")

env_file = load_project_env()
print(f"Loaded environment from: {env_file}")

# Variables cruciales para LangSmith

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGSMITH_API_KEY"] = str(os.getenv("POSTGRESQL_MCP_LANGSMITH") or None)
os.environ["LANGSMITH_PROJECT"] = "Postresql MCP"

os.environ["NVIDIA_API_KEY"] = str(os.getenv("NVIDIA_API_KEY") or None)
os.environ["ORCHESTRATOR_API_KEY_LOCAL"] = str(os.getenv("ORCHESTRATOR_API_KEY_LOCAL") or None)
os.environ["ORCHESTRATOR_BASE_URL_LOCAL"] = str(os.getenv("ORCHESTRATOR_BASE_URL_LOCAL") or None)


from typing import Annotated, TypedDict
from langchain_mcp_adapters.tools import load_mcp_tools
from langgraph.graph.message import add_messages
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver
from langchain_openai import ChatOpenAI

Loaded environment from: /home/santi/Documentos/LangGraph/.env


/home/santi/Documentos/LangGraph/.venv/lib/python3.12/site-packages/langgraph/checkpoint/serde/encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer
/home/santi/Documentos/LangGraph/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
system_env = dict(os.environ)

pg_user = os.getenv("PG_USER")
pg_password = os.getenv("PG_PASSWORD")
pg_host = os.getenv("PG_HOST")
pg_port = os.getenv("PG_PORT") or "5433"
pg_database = os.getenv("PG_DATABASE")

# MCP Client Configuration
client = MultiServerMCPClient(
    {
        "postgresql": {
            "command": "npx",
            "args": [
                "-y",
                "@modelcontextprotocol/server-postgres",
                f"postgresql://{pg_user}:{pg_password}@{pg_host}:{pg_port}/{pg_database}",
            ],
            "transport": "stdio",
            "env": system_env,
        }
    }
)

In [3]:
class State(TypedDict):
    messages: Annotated[list, add_messages]

In [4]:
llm = ChatOpenAI(
    model="z-ai/glm-5.1",
    api_key=os.getenv("NVIDIA_API_KEY"),
    base_url="https://integrate.api.nvidia.com/v1", # NVIDIA's API URL
    temperature=0.0,
)

In [ ]:
from langgraph.prebuilt import tools_condition
from langchain_core.tracers.context import tracing_v2_enabled


def handle_tool_error(error: Exception) -> str:
    return (
        "The database tool failed while executing a query. "
        f"Error detail: {error}. "
        "Do not assume table or column names. Reinspect the schema or metadata, "
        "then retry with a validated query."
    )


async def run_agent():
    async with client.session("postgresql") as mcp_session:
        tools = await load_mcp_tools(mcp_session)
        agent = llm.bind_tools(tools)

        # Call de Agent
        def call_agent(state: State):
            # Get messages from state
            messages = state["messages"]

            # Define Prompt
            agent_prompt = ("system", 
                "You are a expert assistant connected to the SEDICI database through MCP."
                "STRICT RULES: \n"
                "1. NEVER guess or asume the name of the tables, columns or any other database structure."
                "2. Your first step ALWAYS has to be use the tools to explore the database structure and get the information needed to answer the user's question (for example, list_tables or list_resources)."
                "3. Once you have the tables that seems relevant, inspect the structure (tables, types of the columns, etc) before trying to answer the user's question."
                "4. If the query fails, read the error message, understand what went wrong, and use the tools again to get the correct information or fix the query before trying to answer the user's question again."

                "Answer the question based on the provided messages and using the tools when necessary."
            )   

            final_prompt = [agent_prompt] + messages
            # Answer
            response = agent.invoke(final_prompt)

            return {"messages": response}

        # Nodes
        workflow = StateGraph(State)
        workflow.add_node("agent", call_agent)
        workflow.add_node("tools_node", ToolNode(tools, handle_tool_errors=handle_tool_error))

        # Edges
        workflow.add_edge(START, "agent")

        workflow.add_conditional_edges(
            "agent", tools_condition,
            {
                "tools": "tools_node",
                END: END,
            },
        )
        workflow.add_edge("tools_node", "agent")

        # Compile graph
        app = workflow.compile(checkpointer=MemorySaver())

        # Execution
        inputs: State = {
            "messages": [
                ("user", "Calculate the distribution of authors based on the number of words in their names in the SEDICI database. I need a report showing the count of authors grouped by their word count (e.g., how many authors have 2-word names, 3-word names, etc.)")
            ]
        }
        config = {"configurable": {"thread_id": "1"}}

        with tracing_v2_enabled():
            result = await app.ainvoke(inputs, config=config)
            # async for output in app.astream(inputs, config=config):
            #     print(f"Streamed Output: {output}")

        print(result["messages"])

await run_agent()

# Agent Output from Langsmith:

Now I have all the data needed for a comprehensive report. Let me compile the final results.

---

# 📊 Report: Distribution of Authors by Word Count in Their Names

## Data Source

- **Table**: `metadatavalue` in the SEDICI database
- **Primary author field**: `sedici.creator.person` (metadata_field_id = 79) - *"Autor (persona) del documento"*
- **Corporate author field**: `sedici.creator.corporate` (metadata_field_id = 80) - *"Autor (institución) del documento"*
- **Total distinct person authors**: **153,483**
- **Total distinct corporate authors**: **844**

---

## 🔤 Person Authors (`sedici.creator.person`)

| Word Count | # of Distinct Authors | % of Total |
|:----------:|----------------------:|-----------:|
| 1 | 287 | 0.19% |
| 2 | 71,058 | 46.30% |
| 3 | 63,723 | 41.52% |
| 4 | 15,140 | 9.86% |
| 5 | 2,471 | 1.61% |
| 6 | 431 | 0.28% |
| 7 | 137 | 0.09% |
| 8+ | 336 | 0.22% |

### Key Observations

- The vast majority (~97.7%) of person author names have between 2 and 4 words, which aligns with the typical "LastName, FirstName" or "LastName, FirstName MiddleName" format used in SEDICI.
- 2-word names are the most common (46.3%), representing formats like `Salvo, Nahuel` or `González, C.`
- 3-word names are a close second (41.5%), representing formats like `Vila, Gabriela Soledad` or `Romero, Gustavo Esteban`.
- 4-word names (9.9%) typically include compound surnames or multiple middle names, e.g. `Carballeda, Alfredo Juan Manuel` or `Abad, María del Rosario`.

### ⚠️ Data Quality Issues

- **1-word names (287 entries)**: Many are data quality problems - missing commas, garbled text (e.g. `,,,,,`, `213fwerfwe`, `6`, `99`, `Aguil,B.` with missing space after comma).
- **8+ word names (336 entries)**: These are almost entirely cases where multiple authors were concatenated into a single metadata value instead of being entered as separate records. Examples include semicolon-separated lists, pipe-separated lists, or entire author blocks pasted together. For instance, the 325-word entry is actually dozens of authors separated by semicolons and hyphens.

---

## 🏢 Corporate Authors (`sedici.creator.corporate`)

| Word Count | # of Distinct Authors | % of Total |
|:----------:|----------------------:|-----------:|
| 1 | 54 | 6.40% |
| 2 | 94 | 11.14% |
| 3 | 118 | 13.98% |
| 4 | 144 | 17.06% |
| 5 | 114 | 13.51% |
| 6 | 93 | 11.02% |
| 7 | 68 | 8.06% |
| 8 | 66 | 7.82% |
| 9 | 35 | 4.15% |
| 10 | 23 | 2.73% |
| 11+ | 37 | 4.38% |

### Key Observations

- Corporate author names are more evenly distributed across word counts, which is expected since institutional names naturally vary in length (e.g. `Universidad Nacional de La Plata` = 5 words).
- The peak is at 4 words (17.1%), with a fairly flat distribution from 2-8 words.

---

## 📈 Summary Visualization (Person Authors)

```text
Word Count | Author Count (scaled to max)
-----------|-------------------------------------------------------------
    1      | ██
    2      | ████████████████████████████████████████████████████████  ← 71,058
    3      | █████████████████████████████████████████████████████    ← 63,723
    4      | ████████████                                             ← 15,140
    5      | ██                                                       ← 2,471
    6      | ▏                                                        ← 431
    7+     | ▏                                                        ← 337
```

## 💡 Recommendations

1. Clean up 1-word entries - many appear to be malformed (missing spaces after commas, numeric values, or garbage text).
2. Split concatenated author entries - the 336 entries with 8+ words should be parsed and split into individual author records, as they contain multiple authors improperly stored in a single field.
3. Standardize name format - enforce the `LastName, FirstName` convention consistently to improve data quality and discoverability.